### 1. load the data

In [11]:
import pandas as pd
from pathlib import Path

raw = Path("../data/raw")

In [12]:
# table name -> primary key column
tables = {
    "website_sessions": "website_session_id",
    "website_pageviews": "website_pageview_id",
    "orders_mav": "order_id",
    "order_items": "order_item_id",
    "order_item_refunds": "order_item_refund_id",
    "products": "product_id",
}

data = {}

In [13]:
# ---------- 1. Structure, missing values, duplicates ----------
for name, pk in tables.items():
    df = pd.read_csv(raw / f"{name}.csv")
    data[name] = df
    print("=" * 60)
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")
    print("\nData types:")
    print(df.dtypes)
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print("\nMissing values:")
    print(missing if len(missing) > 0 else "none")
    print(f"\nFully duplicated rows: {df.duplicated().sum()}")
    print(f"Duplicated primary keys ({pk}): {df[pk].duplicated().sum()}")

website_sessions: 472871 rows, 9 columns

Data types:
website_session_id    int64
created_at              str
user_id               int64
is_repeat_session     int64
utm_source              str
utm_campaign            str
utm_content             str
device_type             str
http_referer            str
dtype: object

Missing values:
utm_source      83328
utm_campaign    83328
utm_content     83328
http_referer    39917
dtype: int64

Fully duplicated rows: 0
Duplicated primary keys (website_session_id): 0
website_pageviews: 1048575 rows, 4 columns

Data types:
website_pageview_id    int64
created_at               str
website_session_id     int64
pageview_url             str
dtype: object

Missing values:
none

Fully duplicated rows: 0
Duplicated primary keys (website_pageview_id): 0
orders_mav: 32313 rows, 8 columns

Data types:
order_id                int64
created_at                str
website_session_id      int64
user_id                 int64
primary_product_id      int64
items_pu

### 2: missing values, duplicates and data types

In [14]:
for name, pk in tables.items():
    df = data[name]
    print("=" * 60)
    print(name)
    print(df.dtypes)
    missing = df.isna().sum()
    print("\nMissing values:")
    print(missing[missing > 0] if missing.sum() > 0 else "none")
    print("Duplicate rows:", df.duplicated().sum())
    print(f"Duplicate {pk}:", df[pk].duplicated().sum())

website_sessions
website_session_id    int64
created_at              str
user_id               int64
is_repeat_session     int64
utm_source              str
utm_campaign            str
utm_content             str
device_type             str
http_referer            str
dtype: object

Missing values:
utm_source      83328
utm_campaign    83328
utm_content     83328
http_referer    39917
dtype: int64
Duplicate rows: 0
Duplicate website_session_id: 0
website_pageviews
website_pageview_id    int64
created_at               str
website_session_id     int64
pageview_url             str
dtype: object

Missing values:
none
Duplicate rows: 0
Duplicate website_pageview_id: 0
orders_mav
order_id                int64
created_at                str
website_session_id      int64
user_id                 int64
primary_product_id      int64
items_purchased         int64
price_usd             float64
cogs_usd              float64
dtype: object

Missing values:
none
Duplicate rows: 0
Duplicate order_id: 0
o

### 3: date ranges

In [15]:
for name, df in data.items():
    d = pd.to_datetime(df["created_at"], format="%m/%d/%Y %H:%M", errors="coerce")
    print(f"{name}: {d.min()} to {d.max()} | unreadable: {d.isna().sum()}")

website_sessions: 2012-03-19 08:04:00 to 2015-03-19 07:59:00 | unreadable: 0
website_pageviews: 2012-03-19 08:04:00 to 2015-01-16 19:03:00 | unreadable: 0
orders_mav: 2012-03-19 10:42:00 to 2015-03-19 05:38:00 | unreadable: 0
order_items: 2012-03-19 10:42:00 to 2015-03-19 05:38:00 | unreadable: 0
order_item_refunds: 2012-04-06 11:32:00 to 2015-04-01 18:11:00 | unreadable: 0
products: 2012-03-19 08:00:00 to 2014-02-05 10:00:00 | unreadable: 0


### Cell 4: broken links between tables (all should be 0)

In [17]:
sessions = data["website_sessions"]
pageviews = data["website_pageviews"]
orders = data["orders_mav"]
items = data["order_items"]
refunds = data["order_item_refunds"]
products = data["products"]

print("orders with no session:", (~orders["website_session_id"].isin(sessions["website_session_id"])).sum())
print("order_items with no order:", (~items["order_id"].isin(orders["order_id"])).sum())
print("order_items with no product:", (~items["product_id"].isin(products["product_id"])).sum())
print("refunds with no order_item:", (~refunds["order_item_id"].isin(items["order_item_id"])).sum())
print("pageviews with no session:", (~pageviews["website_session_id"].isin(sessions["website_session_id"])).sum())

orders with no session: 0
order_items with no order: 0
order_items with no product: 0
refunds with no order_item: 0
pageviews with no session: 0


### 5: do the money totals agree?

In [18]:
print("Revenue in orders_mav :", round(orders["price_usd"].sum(), 2))
print("Revenue in order_items:", round(items["price_usd"].sum(), 2))
print("Total refunds         :", round(refunds["refund_amount_usd"].sum(), 2))

Revenue in orders_mav : 1938509.75
Revenue in order_items: 1938509.75
Total refunds         : 85338.69


### 6: category values

In [19]:
for col in ["utm_source", "utm_campaign", "utm_content", "device_type", "is_repeat_session"]:
    print(f"\n{col}")
    print(sessions[col].value_counts(dropna=False))


utm_source
utm_source
gsearch       316035
NaN            83328
bsearch        62823
socialbook     10685
Name: count, dtype: int64

utm_campaign
utm_campaign
nonbrand            337615
NaN                  83328
brand                41243
desktop_targeted      5590
pilot                 5095
Name: count, dtype: int64

utm_content
utm_content
g_ad_1         282706
NaN             83328
b_ad_1          54909
g_ad_2          33329
b_ad_2           7914
social_ad_2      5590
social_ad_1      5095
Name: count, dtype: int64

device_type
device_type
desktop    327027
mobile     145844
Name: count, dtype: int64

is_repeat_session
is_repeat_session
0    394318
1     78553
Name: count, dtype: int64


### A: summary of all tables

In [20]:
for name, df in data.items():
    miss = df.isna().sum()
    print(name, "| rows:", len(df), "| duplicate rows:", df.duplicated().sum(), "| missing:", miss[miss > 0].to_dict())

website_sessions | rows: 472871 | duplicate rows: 0 | missing: {'utm_source': 83328, 'utm_campaign': 83328, 'utm_content': 83328, 'http_referer': 39917}
website_pageviews | rows: 1048575 | duplicate rows: 0 | missing: {}
orders_mav | rows: 32313 | duplicate rows: 0 | missing: {}
order_items | rows: 40025 | duplicate rows: 0 | missing: {}
order_item_refunds | rows: 1731 | duplicate rows: 0 | missing: {}
products | rows: 4 | duplicate rows: 0 | missing: {}


### B: device types and the blank-source check

In [21]:
print(sessions["device_type"].value_counts(dropna=False))
print("Blank source AND blank referer:", (sessions["utm_source"].isna() & sessions["http_referer"].isna()).sum())
print("Blank source BUT has referer:", (sessions["utm_source"].isna() & sessions["http_referer"].notna()).sum())
print(sessions.loc[sessions["utm_source"].isna(), "http_referer"].value_counts().head())

device_type
desktop    327027
mobile     145844
Name: count, dtype: int64
Blank source AND blank referer: 39917
Blank source BUT has referer: 43411
http_referer
https://www.gsearch.com    35202
https://www.bsearch.com     8209
Name: count, dtype: int64
